# Lecture 11 · Notebook 3 — LoRA parameter counts and where training memory goes
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_3_lora_and_training_memory.ipynb)

**Companion to [Chapter 6](https://nlp-course-cuhksz.github.io/lecture11.html#p6-s81)** (LoRA, slides 78–81) and **[Chapter 7](https://nlp-course-cuhksz.github.io/lecture11.html#p7-s107)** (memory accounting and ZeRO,
slides 106–111) of the video lecture. In this notebook you will:

1. count LoRA's trainable parameters for one matrix and for a whole 7B model,
2. redo the lecture's memory accounting: 16 bytes per parameter for mixed-precision Adam, 112 GB for a 7B model (slide 108),
   and what LoRA and QLoRA leave of it,
3. reproduce the ZeRO paper's 120 / 31.4 / 16.6 / 1.9 GB per GPU (slides 109–110),
4. write LoRA **from scratch** in a few lines, attach it to SmolLM2-135M, check that it starts as an exact copy of the base model,
   fine-tune it on CPU, and **merge** $BA$ back into $W$ for zero extra latency.

Runs on **CPU in about 5–10 minutes** (download ~270 MB).

In [ ]:
!pip -q install -U transformers

In [ ]:
import json, math, time, torch
from huggingface_hub import hf_hub_download

import matplotlib.pyplot as plt
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "grid.linewidth": 0.8, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "font.size": 11, "lines.linewidth": 2})

torch.manual_seed(0)

## 1 · One matrix
LoRA replaces the update $\Delta W$ (size $d \times k$) by $BA$ with $B \in \mathbb{R}^{d\times r}$, $A \in \mathbb{R}^{r\times k}$ (slide 80).
Trainable parameters: $r(d + k)$ instead of $dk$. Slide 81: $d = k = 4096$, $r = 8$.

In [ ]:
d = k = 4096
print(f"full update: {d * k:,} parameters")
for r in [1, 4, 8, 16, 64, 256]:
    print(f"rank {r:3d}: {r * (d + k):9,} parameters = {r * (d + k) / (d * k):6.2%} of the matrix")

## 2 · A whole model
Real recipes put LoRA on several matrices per block. We read Qwen2.5-7B's config and count LoRA parameters for two common choices:
only the query and value projections (as in the LoRA paper), or all seven linear layers (attention q, k, v, o and FFN gate, up, down).

In [ ]:
c = json.load(open(hf_hub_download("Qwen/Qwen2.5-7B-Instruct", "config.json")))
D, F, L = c["hidden_size"], c["intermediate_size"], c["num_hidden_layers"]
hd = D // c["num_attention_heads"]; kvd = c["num_key_value_heads"] * hd
shapes = {"q": (D, D), "k": (D, kvd), "v": (D, kvd), "o": (D, D), "gate": (D, F), "up": (D, F), "down": (F, D)}   # (in, out)
total = json.load(open(hf_hub_download("Qwen/Qwen2.5-7B-Instruct", "model.safetensors.index.json")))["metadata"]["total_size"] / 2
print(f"Qwen2.5-7B: {L} layers, hidden {D}, FFN {F}, K/V width {kvd} (GQA) -> {total / 1e9:.2f}B parameters\n")
def lora_params(targets, r):
    return L * sum(r * (shapes[t][0] + shapes[t][1]) for t in targets)
for targets in [["q", "v"], list(shapes)]:
    for r in [8, 16, 64]:
        n = lora_params(targets, r)
        print(f"LoRA on {'+'.join(targets):22s} rank {r:2d}: {n / 1e6:7.1f}M trainable = {n / total:.3%} of the model")

## 3 · Where training memory goes
Mixed-precision Adam keeps, per parameter (slide 107): FP16 weights (2 bytes) + FP16 gradients (2) + FP32 master weights, momentum and
variance (4 + 4 + 4) = **16 bytes**, before activations. With LoRA the base model is frozen: it needs only its FP16 weights — no
gradients, no optimizer states — and the 16 bytes apply to the small adapters only. QLoRA stores the frozen base in 4 bits
(about 0.5 byte per parameter; we ignore the small overhead of the quantization constants).

In [ ]:
def full_ft(P):            return 16 * P
def lora(P, n_lora):       return 2 * P + 16 * n_lora
def qlora(P, n_lora):      return 0.5 * P + 16 * n_lora

P = 7e9                                 # HuatuoGPT-7B on slide 108
n = lora_params(list(shapes), 16)       # a rank-16 LoRA on all linear layers of a 7B model
parts = {"FP16 weights": 2 * P, "FP16 gradients": 2 * P, "FP32 master weights": 4 * P, "Adam momentum m": 4 * P, "Adam variance v": 4 * P}
for k_, v_ in parts.items():
    print(f"  {k_:20s} {v_ / 1e9:6.0f} GB")
print(f"full fine-tuning  {full_ft(P) / 1e9:6.0f} GB  (one 80 GB GPU is not enough)")
print(f"LoRA r=16         {lora(P, n) / 1e9:6.1f} GB")
print(f"QLoRA r=16        {qlora(P, n) / 1e9:6.1f} GB  (plus activations)")

fig, ax = plt.subplots(figsize=(7.5, 2.6))
vals = [full_ft(P) / 1e9, lora(P, n) / 1e9, qlora(P, n) / 1e9]
labels = ["full fine-tuning", "LoRA (r=16)", "QLoRA (r=16)"]
ax.barh(labels[::-1], vals[::-1], color=BLUE, height=0.5)
for i, v in enumerate(vals[::-1]):
    ax.text(v + 1.5, i, f"{v:.1f} GB", va="center")
ax.axvline(80, color=ORANGE, lw=1.5); ax.text(80, 2.45, " one 80 GB GPU", color=INK, va="center")
ax.set_xlabel("training states for a 7B model (GB, without activations)"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 4 · ZeRO: stop storing the same thing on every GPU
In data parallelism every GPU holds all 16 bytes per parameter. ZeRO shards the optimizer states (stage 1), then also the gradients
(stage 2), then also the weights (stage 3) across the $N_d$ GPUs (slide 110). With $\Psi$ parameters and $K = 12$:

In [ ]:
def zero_gb(psi, nd, K=12):
    return {"data parallel": (2 + 2 + K) * psi,
            "ZeRO-1 (P_os)": 2 * psi + 2 * psi + K * psi / nd,
            "ZeRO-2 (P_os+g)": 2 * psi + (2 + K) * psi / nd,
            "ZeRO-3 (P_os+g+p)": (2 + 2 + K) * psi / nd}

for k_, v_ in zero_gb(7.5e9, 64).items():
    print(f"7.5B model, 64 GPUs, {k_:18s}: {v_ / 1e9:6.1f} GB per GPU")
print()
for psi, nd in [(7.5e9, 4), (128e9, 64), (1e12, 1024)]:
    print(f"ZeRO-3, {psi / 1e9:6.1f}B parameters on {nd:5d} GPUs: {zero_gb(psi, nd)['ZeRO-3 (P_os+g+p)'] / 1e9:6.1f} GB per GPU")

These match slide 111's bold entries (30 GB, 32 GB, 15.6 GB per GPU). Stages 1 and 2 cost no extra communication compared with plain data
parallelism; stage 3 costs about 1.5× because weights must be gathered in both the forward and the backward pass.

## 5 · LoRA from scratch
A LoRA layer wraps a frozen `nn.Linear`: $h = W_0x + \frac{\alpha}{r} B A x$, with $A$ random and $B = 0$, so it starts as an exact copy
of the pre-trained layer (slide 80). We attach it to the query and value projections of every block of SmolLM2-135M.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
SMOL = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(SMOL)
model = AutoModelForCausalLM.from_pretrained(SMOL, dtype=torch.float32)

class LoRALinear(torch.nn.Module):
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = torch.nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

probe = tok("Where does the time and memory go?", return_tensors="pt").input_ids
with torch.no_grad():
    before = model(probe).logits

for p in model.parameters():
    p.requires_grad_(False)                      # freeze the whole base model
for layer in model.model.layers:
    layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj)
    layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,} parameters = {trainable / total:.2%}")
with torch.no_grad():
    print("max |logit change| at initialisation:", (model(probe).logits - before).abs().max().item())

Now fine-tune only $A$ and $B$ on a tiny made-up dataset: short question–answer pairs about this lecture. The base model has never seen
this format; after a few minutes of training on CPU it answers in it.

In [ ]:
pairs = [
    ("What does LoRA freeze?", "The pre-trained weight W. Only the thin matrices B and A are trained."),
    ("Why is decoding slow?", "It is memory-bound: every token reads all weights and the KV cache."),
    ("What does GQA share?", "Groups of query heads share one key head and one value head."),
    ("What does FlashAttention avoid?", "Writing the N x N attention matrix to slow GPU memory."),
    ("What does ZeRO shard?", "Optimizer states, gradients and weights across data-parallel GPUs."),
    ("What does speculative decoding need?", "A small draft model and one verification pass of the big model."),
    ("What does PagedAttention borrow?", "Virtual memory: the KV cache is stored in small fixed-size blocks."),
    ("What does MoE activate?", "Only a few experts per token, chosen by a router."),
    ("What does quantization change?", "The number of bits used to store each weight."),
    ("What does distillation train?", "A small student to imitate a large teacher."),
    ("What limits the batch size when serving?", "The KV cache, because each request has its own."),
    ("Where does the time go?", "Often to bytes moved, not to arithmetic."),
]
texts = [f"Q: {q}\nA: {a}{tok.eos_token}" for q, a in pairs]

@torch.no_grad()
def answer(q, n=24):
    enc = tok(f"Q: {q}\nA:", return_tensors="pt")
    out = model.generate(**enc, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).split("\n")[0].strip()

print("before:", answer("What does LoRA freeze?"))

tok.pad_token = tok.eos_token
batch = tok(texts, return_tensors="pt", padding=True)
labels = batch.input_ids.masked_fill(batch.attention_mask == 0, -100)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-3)
model.train(); losses = []; t0 = time.time()
for step in range(60):
    loss = model(**batch, labels=labels).loss
    loss.backward(); opt.step(); opt.zero_grad()
    losses.append(loss.item())
model.eval()
print(f"60 steps in {time.time() - t0:.0f} s, loss {losses[0]:.2f} -> {losses[-1]:.2f}")
for q in ["What does LoRA freeze?", "What does GQA share?", "Where does the time go?"]:
    print(f"Q: {q}\nA: {answer(q)}")

fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(losses, color=BLUE)
ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.set_title("LoRA fine-tuning, rank 8 on q and v", loc="left")
plt.tight_layout(); plt.show()

## 6 · Merge $BA$ into $W$: zero extra latency
After training we can fold the update into the frozen weight, $W \leftarrow W_0 + \frac{\alpha}{r}BA$, and throw the adapters away (slide 81).
The merged model is an ordinary model of the original size and speed, and it gives the same outputs up to floating-point rounding. (Or keep the small $A$, $B$ files
separately and swap them per task or per user on one shared base model.)

In [ ]:
with torch.no_grad():
    lora_logits = model(probe).logits
    for layer in model.model.layers:
        for name in ["q_proj", "v_proj"]:
            m = getattr(layer.self_attn, name)
            m.base.weight += (m.B @ m.A) * m.scale
            setattr(layer.self_attn, name, m.base)
    merged_logits = model(probe).logits
print("max |difference| between LoRA model and merged model:", (merged_logits - lora_logits).abs().max().item())
print("parameters after merging:", f"{sum(p.numel() for p in model.parameters()):,}")
print("A: " + answer("What does LoRA freeze?"))

## Exercises
1. Change the rank to 1 and to 64. How do the trainable parameters, the final loss and the answers change?
2. Put LoRA on all seven linear layers instead of q and v only. How many parameters is that for SmolLM2-135M?
3. With section 3's functions: what is the smallest number of 80 GB GPUs that can fully fine-tune a 70B model with ZeRO-3
   (ignore activations)? How does that compare with QLoRA on one GPU?
4. Ask a question that is **not** in the training set. Does the model generalise the format, the content, or neither?

**Back to the [video lecture](https://nlp-course-cuhksz.github.io/lecture11.html)** · Previous: [Notebook 1 — the KV cache](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_1_kv_cache_and_gqa.ipynb), [Notebook 2 — quantization](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture11/lecture11_2_quantize_a_small_model.ipynb)